In [1]:
from generate_stream import generate_qubit_stream, animate_qubit_drift
import torch
from torch.nn import CrossEntropyLoss
from torch.optim import SGD

# Strictly 0.6.0 Imports
from avalanche.benchmarks import tensors_benchmark
from avalanche.models import SimpleMLP
from avalanche.training.supervised import Naive, Replay
from avalanche.training.plugins import EvaluationPlugin
from avalanche.evaluation.metrics import accuracy_metrics, loss_metrics
from avalanche.logging import InteractiveLogger, TensorboardLogger


data_stream = generate_qubit_stream(seed=2026)
# 1. PREPARE THE DATA (80% Train / 20% Test per window)
train_x, train_y = [], []
test_x, test_y = [], []

for e in data_stream:
    x = torch.from_numpy(e['x']).float()
    y = torch.from_numpy(e['y']).long()
    # Split 5000 shots into 4000 train / 1000 test
    split_idx = int(len(x) * 0.8)
    train_x.append(x[:split_idx])
    train_y.append(y[:split_idx])
    test_x.append(x[split_idx:])
    test_y.append(y[split_idx:])




In [3]:
len(data_stream), len(train_x), len(train_y), len(test_x), len(test_y)

(336, 336, 336, 336, 336)

In [17]:
len(train_x), len(train_y), len(test_x), len(test_y)
train_x[0].shape, train_y[0].shape, test_x[0].shape, test_y[0].shape
zeros_train_y_0 = (train_y[0] == 0).sum()
ones_train_y_0 = (train_y[0] == 1).sum()
ones_train_y_0

tensor(1500)

In [ ]:
# Create the scenario. CRITICAL: task_labels must be provided.
# No shuffling to preserve the 7-day chronological drift.
scenario = tensors_benchmark(
    train_tensors=list(zip(train_x, train_y)),
    test_tensors=list(zip(test_x, test_y)),
    task_labels=[0] * len(data_stream)
)

def run_qubit_experiment(strategy_name):
    print(f"--- Starting Experiment: {strategy_name} ---")
    
    # LOGGERS: Tensorboard for graphs, Interactive for clean terminal
    # InteractiveLogger: we set it to quiet mode by limiting metric frequency
    tb_logger = TensorboardLogger(f"logs/{strategy_name}")
    int_logger = InteractiveLogger()
    
    # EVALUATION PLUGIN: We only track Experience accuracy to keep it "quiet"
    # Setting minibatch=False and epoch=False removes the wordy output
    eval_plugin = EvaluationPlugin(
        accuracy_metrics(minibatch=False, epoch=False, experience=True, stream=True),
        loss_metrics(minibatch=False, epoch=False, experience=True, stream=True),
        loggers=[int_logger, tb_logger]
    )
    
    model = SimpleMLP(input_size=2, num_classes=2, hidden_size=16)
    
    if strategy_name == 'naive':
        strategy = Naive(
            model, SGD(model.parameters(), lr=0.01), CrossEntropyLoss(),
            train_mb_size=64, train_epochs=1, evaluator=eval_plugin
        )
    else:
        # Replay uses a small buffer (200) to track the rotation
        strategy = Replay(
            model, SGD(model.parameters(), lr=0.01), CrossEntropyLoss(),
            mem_size=200, train_mb_size=64, train_epochs=1, evaluator=eval_plugin
        )

    # TRAINING & EVALUATION STREAM
    for i, exp in enumerate(scenario.train_stream):
        # 1. Feed-while-training: Train on the current 30-min window
        strategy.train(exp)
        
        # 2. Evaluate ONLY on the corresponding test window to check drift tracking
        strategy.eval(scenario.test_stream[i])


In [2]:
# Run comparison
run_qubit_experiment('naive')
run_qubit_experiment('replay')

/Users/chi/Documents/ContinualLearning/.venv/lib/python3.10/site-packages/avalanche/training/templates/base.py:468: PositionalArgumentsDeprecatedWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your are passing 3 positional arguments to the Naive.__init__ method. Consider passing them as names arguments. The ability to pass positional arguments will be removed in the future.
  warnings.warn(error_str, category=PositionalArgumentsDeprecatedWarning)


--- Starting Experiment: naive ---
-- >> Start of training phase << --
100%|██████████| 63/63 [00:00<00:00, 368.90it/s]
Epoch 0 ended.
-- >> End of training phase << --
-- >> Start of eval phase << --
-- Starting eval on experience 0 (Task 0) from test stream --
100%|██████████| 16/16 [00:00<00:00, 501.18it/s]
> Eval on experience 0 (Task 0) from test stream ended.
	Loss_Exp/eval_phase/test_stream/Task000/Exp000 = 0.6843
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp000 = 0.6150
-- >> End of eval phase << --
	Loss_Stream/eval_phase/test_stream/Task000 = 0.6843
	Top1_Acc_Stream/eval_phase/test_stream/Task000 = 0.6150
-- >> Start of training phase << --
100%|██████████| 63/63 [00:00<00:00, 450.50it/s]
Epoch 0 ended.
-- >> End of training phase << --
-- >> Start of eval phase << --
-- Starting eval on experience 1 (Task 0) from test stream --
100%|██████████| 16/16 [00:00<00:00, 716.05it/s]
> Eval on experience 1 (Task 0) from test stream ended.
	Loss_Exp/eval_phase/test_stream/Task000/

/Users/chi/Documents/ContinualLearning/.venv/lib/python3.10/site-packages/avalanche/training/templates/base.py:468: PositionalArgumentsDeprecatedWarning: Avalanche is transitioning to strategy constructors that accept named (keyword) arguments only. This is done to ensure that there is no confusion regarding the meaning of each argument (strategies can have many arguments). Your are passing 3 positional arguments to the Replay.__init__ method. Consider passing them as names arguments. The ability to pass positional arguments will be removed in the future.
  warnings.warn(error_str, category=PositionalArgumentsDeprecatedWarning)


-- >> Start of training phase << --
100%|██████████| 63/63 [00:00<00:00, 555.93it/s]

/Users/chi/Documents/ContinualLearning/.venv/lib/python3.10/site-packages/avalanche/training/plugins/replay.py:123: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)


...


Epoch 0 ended.
-- >> End of training phase << --
-- >> Start of eval phase << --
-- Starting eval on experience 333 (Task 0) from test stream --
100%|██████████| 16/16 [00:00<00:00, 755.39it/s]
> Eval on experience 333 (Task 0) from test stream ended.
	Loss_Exp/eval_phase/test_stream/Task000/Exp333 = 0.1856
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp333 = 0.9340
-- >> End of eval phase << --
	Loss_Stream/eval_phase/test_stream/Task000 = 0.1856
	Top1_Acc_Stream/eval_phase/test_stream/Task000 = 0.9340
-- >> Start of training phase << --
100%|██████████| 63/63 [00:00<00:00, 369.85it/s]

/Users/chi/Documents/ContinualLearning/.venv/lib/python3.10/site-packages/avalanche/training/plugins/replay.py:123: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)



Epoch 0 ended.
-- >> End of training phase << --
-- >> Start of eval phase << --
-- Starting eval on experience 334 (Task 0) from test stream --
100%|██████████| 16/16 [00:00<00:00, 746.97it/s]
> Eval on experience 334 (Task 0) from test stream ended.
	Loss_Exp/eval_phase/test_stream/Task000/Exp334 = 0.2081
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp334 = 0.9230
-- >> End of eval phase << --
	Loss_Stream/eval_phase/test_stream/Task000 = 0.2081
	Top1_Acc_Stream/eval_phase/test_stream/Task000 = 0.9230
-- >> Start of training phase << --
100%|██████████| 63/63 [00:00<00:00, 368.02it/s]

/Users/chi/Documents/ContinualLearning/.venv/lib/python3.10/site-packages/avalanche/training/plugins/replay.py:123: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)



Epoch 0 ended.
-- >> End of training phase << --
-- >> Start of eval phase << --
-- Starting eval on experience 335 (Task 0) from test stream --
100%|██████████| 16/16 [00:00<00:00, 732.60it/s]
> Eval on experience 335 (Task 0) from test stream ended.
	Loss_Exp/eval_phase/test_stream/Task000/Exp335 = 0.1899
	Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp335 = 0.9350
-- >> End of eval phase << --
	Loss_Stream/eval_phase/test_stream/Task000 = 0.1899
	Top1_Acc_Stream/eval_phase/test_stream/Task000 = 0.9350
